<a href="https://colab.research.google.com/github/Trinesh200542/Generative-AI-SDC/blob/main/FineTuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Step 1: Install required packages
import os
os.environ["WANDB_DISABLED"] = "true"
!pip install transformers datasets torch
# Step 0: Disable wandb


# Step 1: Install required packages


# Step 2: Import necessary libraries
from transformers import GPT2Tokenizer, GPT2LMHeadModel, Trainer, TrainingArguments
from datasets import load_dataset, Dataset
import torch
from torch.utils.data import Dataset as TorchDataset
import random

# [Rest of your original code remains the same...]
# Step 2: Import necessary libraries
from transformers import GPT2Tokenizer, GPT2LMHeadModel, Trainer, TrainingArguments
from datasets import load_dataset, Dataset
import torch
from torch.utils.data import Dataset as TorchDataset
import random

# Step 3: Load and prepare the dataset
# We'll use a sample of Amazon product reviews
print("Loading and preparing dataset...")

# Create a small custom dataset of product reviews
reviews = [
    "This product is amazing! It works perfectly and exceeded my expectations.",
    "The quality is poor and it broke after just two days of use.",
    "Excellent value for the price. I would definitely buy this again.",
    "Not what I expected. The description was misleading.",
    "Fast shipping and great customer service. The product itself is just okay.",
    "This is the best purchase I've made all year. Highly recommended!",
    "Terrible experience. The item arrived damaged and the seller was unresponsive.",
    "Good product overall, but the instructions could be clearer.",
    "Works as advertised. No complaints here.",
    "I'm very satisfied with this purchase. It does exactly what it's supposed to do."
]

# Create a Hugging Face Dataset
dataset = Dataset.from_dict({"text": reviews})

# Split dataset into train and test
dataset = dataset.train_test_split(test_size=0.2)
train_dataset = dataset["train"]
test_dataset = dataset["test"]

# Step 4: Load tokenizer and model
print("Loading tokenizer and model...")
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token  # Set pad token

model = GPT2LMHeadModel.from_pretrained("gpt2")

# Step 5: Tokenize the dataset
def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, padding="max_length", max_length=64)

print("Tokenizing dataset...")
tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_test = test_dataset.map(tokenize_function, batched=True)

# Step 6: Prepare PyTorch datasets
class ReviewDataset(TorchDataset):
    def __init__(self, tokenized_dataset):
        self.input_ids = tokenized_dataset["input_ids"]
        self.attention_mask = tokenized_dataset["attention_mask"]

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return {
            "input_ids": torch.tensor(self.input_ids[idx]),
            "attention_mask": torch.tensor(self.attention_mask[idx]),
            "labels": torch.tensor(self.input_ids[idx])  # For language modeling, labels are input_ids
        }

train_pt = ReviewDataset(tokenized_train)
test_pt = ReviewDataset(tokenized_test)

# Step 7: Set up training arguments
training_args = TrainingArguments(
    output_dir="./gpt2_finetuned",
    overwrite_output_dir=True,
    num_train_epochs=20,
    per_device_train_batch_size=2,
    save_steps=10_000,
    save_total_limit=2,
    prediction_loss_only=True,
    logging_steps=100,
)

# Step 8: Create Trainer and fine-tune
print("Starting fine-tuning...")
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_pt,
    eval_dataset=test_pt,
)

trainer.train()

# Step 9: Save the fine-tuned model
print("Saving fine-tuned model...")
model.save_pretrained("./gpt2_finetuned")
tokenizer.save_pretrained("./gpt2_finetuned")

# Step 10: Test the fine-tuned model
print("\nTesting the fine-tuned model...")

# Load the fine-tuned model
model = GPT2LMHeadModel.from_pretrained("./gpt2_finetuned")
model.eval()

# Generate some sample text
def generate_sample(prompt):
    inputs = tokenizer(prompt, return_tensors="pt")
    output = model.generate(
        inputs.input_ids,
        max_length=50,
        num_return_sequences=1,
        no_repeat_ngram_size=2,
        do_sample=True,
        top_k=50,
        top_p=0.95,
        temperature=0.7
    )
    return tokenizer.decode(output[0], skip_special_tokens=True)

# Test with product review prompts
test_prompts = [
    "This product is",
    "I would recommend",
    "The quality is",
    "Terrible experience",
    "Excellent value"
]

print("\nGenerated product reviews:")
for prompt in test_prompts:
    generated = generate_sample(prompt)
    print(f"\nPrompt: '{prompt}'\nGenerated: {generated}\n{'-'*50}")

Loading and preparing dataset...
Loading tokenizer and model...
Tokenizing dataset...


Map:   0%|          | 0/8 [00:00<?, ? examples/s]

Map:   0%|          | 0/2 [00:00<?, ? examples/s]

Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


Starting fine-tuning...


...


Prompt: 'I would recommend'
Generated: I would recommend this product to anyone. It does exactly what it's supposed to do.
--------------------------------------------------


The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.



Prompt: 'The quality is'
Generated: The quality is poor and it broke after just two days of use.
--------------------------------------------------


The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.



Prompt: 'Terrible experience'
Generated: Terrible experience with this purchase. I would definitely buy this again.
--------------------------------------------------

Prompt: 'Excellent value'
Generated: Excellent value for the price. I would definitely buy this again.
--------------------------------------------------
